In [1]:
# Install if needed:
# !pip install pandas numpy plotly ipywidgets python-pptx kaleido openpyxl

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML
import ipywidgets as widgets
from pptx import Presentation
from pptx.util import Inches

# -----------------------------
# 1. CREATE DATASET
# -----------------------------
np.random.seed(42)

dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
regions = ["North", "South", "East", "West"]
categories = ["Electronics", "Furniture", "Office Supplies", "Clothing"]
products = {
    "Electronics": ["Laptop", "Phone", "Tablet"],
    "Furniture": ["Chair", "Desk", "Sofa"],
    "Office Supplies": ["Notebook", "Pen", "Printer"],
    "Clothing": ["Shirt", "Jeans", "Jacket"]
}

rows = []

for _ in range(3000):
    date = np.random.choice(dates)
    region = np.random.choice(regions)
    category = np.random.choice(categories)
    product = np.random.choice(products[category])
    units = np.random.randint(1, 10)
    price = np.random.randint(20, 900)
    discount = np.random.choice([0, 0.05, 0.10, 0.15, 0.20])
    revenue = units * price * (1 - discount)
    cost = revenue * np.random.uniform(0.55, 0.80)
    profit = revenue - cost

    rows.append([date, region, category, product, units, price, discount, revenue, cost, profit])

df = pd.DataFrame(rows, columns=[
    "Date", "Region", "Category", "Product", "Units", "Price",
    "Discount", "Revenue", "Cost", "Profit"
])

df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Profit Margin"] = df["Profit"] / df["Revenue"]

df.to_csv("week6_business_sales_dataset.csv", index=False)
df.head()

,Date,Region,Category,Product,Units,Price,Discount,Revenue,Cost,Profit,Month,Profit Margin
0,2024-04-12,West,Electronics,Tablet,8,720,0.20,4608.00,2714.133474,1893.866526,2024-04,0.410995
1,2024-08-02,East,Office Supplies,Notebook,4,891,0.10,3207.60,1780.686706,1426.913294,2024-08,0.444854
2,2024-09-14,West,Clothing,Jeans,6,405,0.15,2065.50,1230.730504,834.769496,2024-09,0.404149
3,2024-11-09,South,Electronics,Laptop,1,494,0.10,444.60,288.974547,155.625453,2024-11,0.350035
4,2024-07-06,West,Office Supplies,Pen,3,895,0.15,2282.25,1593.247037,689.002963,2024-07,0.301896


In [3]:
# -----------------------------
# 2. DASHBOARD WITH FILTERS
# -----------------------------
region_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Region"].unique()),
    value="All",
    description="Region:"
)

category_filter = widgets.Dropdown(
    options=["All"] + sorted(df["Category"].unique()),
    value="All",
    description="Category:"
)

def update_dashboard(region, category):
    data = df.copy()

    if region != "All":
        data = data[data["Region"] == region]

    if category != "All":
        data = data[data["Category"] == category]

    total_revenue = data["Revenue"].sum()
    total_profit = data["Profit"].sum()
    total_units = data["Units"].sum()
    avg_margin = data["Profit Margin"].mean()

    display(HTML(f"""
    <h2>Week 6 Sales Performance Dashboard</h2>
    <h3>Key Business Metrics</h3>
    <ul>
        <li><b>Total Revenue:</b> ${total_revenue:,.0f}</li>
        <li><b>Total Profit:</b> ${total_profit:,.0f}</li>
        <li><b>Total Units Sold:</b> {total_units:,}</li>
        <li><b>Average Profit Margin:</b> {avg_margin:.1%}</li>
    </ul>
    """))

    monthly = data.groupby("Month", as_index=False)[["Revenue", "Profit"]].sum()
    fig1 = px.line(
        monthly,
        x="Month",
        y=["Revenue", "Profit"],
        title="Monthly Revenue and Profit Trend",
        markers=True
    )
    fig1.show()

    category_sales = data.groupby("Category", as_index=False)["Revenue"].sum()
    fig2 = px.bar(
        category_sales,
        x="Category",
        y="Revenue",
        title="Revenue by Product Category",
        text_auto=".2s"
    )
    fig2.show()

    region_profit = data.groupby("Region", as_index=False)["Profit"].sum()
    fig3 = px.bar(
        region_profit,
        x="Region",
        y="Profit",
        title="Profit by Region",
        text_auto=".2s"
    )
    fig3.show()

    product_sales = data.groupby("Product", as_index=False)["Revenue"].sum().sort_values(
        "Revenue", ascending=False
    )
    fig4 = px.treemap(
        product_sales,
        path=["Product"],
        values="Revenue",
        title="Drill-Down View: Revenue by Product"
    )
    fig4.show()

widgets.interact(update_dashboard, region=region_filter, category=category_filter)

interactive(children=(Dropdown(description='Region:', options=('All', 'East', 'North', 'South', 'West'), value…

<function __main__.update_dashboard(region, category)>

In [5]:
# -----------------------------
# 3. BUSINESS INSIGHTS
# -----------------------------
summary = df.groupby("Category")[["Revenue", "Profit"]].sum().sort_values("Revenue", ascending=False)
best_category = summary.index[0]

region_summary = df.groupby("Region")["Profit"].sum().sort_values(ascending=False)
best_region = region_summary.index[0]

monthly_summary = df.groupby("Month")["Revenue"].sum()
best_month = monthly_summary.idxmax()

insights = [
    f"{best_category} generated the highest revenue overall.",
    f"{best_region} was the most profitable region.",
    f"{best_month} was the strongest sales month.",
    "Revenue and profit generally move together, showing healthy cost control.",
    "Filters help managers compare regions, categories, and products quickly."
]

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

1. Furniture generated the highest revenue overall.
2. South was the most profitable region.
3. 2024-10 was the strongest sales month.
4. Revenue and profit generally move together, showing healthy cost control.
5. Filters help managers compare regions, categories, and products quickly.
